# 🤖 Fase 4: Modeling & Evaluation
**Geo-Price Analyzer** — Membandingkan Linear Regression, Random Forest, XGBoost

---
**Target:** R² Score > 0.80

In [ ]:
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'jabodetabek_processed.csv'))
print(f'📂 Data: {df.shape[0]} baris × {df.shape[1]} kolom')

# Recreate LabelEncoder dari kolom city (agar bisa disimpan)
le = LabelEncoder()
le.fit(df['city'])
print(f'📋 City classes: {list(le.classes_)}')

feature_cols = ['land_size_m2','building_size_m2','bedrooms','bathrooms',
                'carports','garages','floors','city_encoded',
                'rasio_tanah_bangunan','total_ruangan']
X = df[feature_cols]
y = df['price_in_rp']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 4.1 Load Data Processed

In [ ]:
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'jabodetabek_processed.csv'))
print(f'📂 Data: {df.shape[0]} baris × {df.shape[1]} kolom')

feature_cols = ['land_size_m2','building_size_m2','bedrooms','bathrooms',
                'carports','garages','floors','city_encoded',
                'rasio_tanah_bangunan','total_ruangan']
X = df[feature_cols]
y = df['price_in_rp']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 4.2 Model 1: Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_r2 = r2_score(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae = mean_absolute_error(y_test, lr_pred)
print(f'R²={lr_r2:.4f} | RMSE={format_rupiah(lr_rmse)} | MAE={format_rupiah(lr_mae)}')

## 4.3 Model 2: Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f'R²={rf_r2:.4f} | RMSE={format_rupiah(rf_rmse)} | MAE={format_rupiah(rf_mae)}')

## 4.4 Model 3: XGBoost

In [ ]:
xgb = XGBRegressor(n_estimators=200, max_depth=7, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_r2 = r2_score(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae = mean_absolute_error(y_test, xgb_pred)
print(f'R²={xgb_r2:.4f} | RMSE={format_rupiah(xgb_rmse)} | MAE={format_rupiah(xgb_mae)}')

## 4.5 Perbandingan Model

In [ ]:
comp = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'R²': [lr_r2, rf_r2, xgb_r2],
    'RMSE': [lr_rmse, rf_rmse, xgb_rmse],
    'MAE': [lr_mae, rf_mae, xgb_mae]
})
best = comp.loc[comp['R²'].idxmax(), 'Model']
print(f'🏆 Model Terbaik: {best}\n')
comp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e74c3c','#2ecc71','#3498db']
for i, (metric, key) in enumerate(zip(['R² Score','RMSE','MAE'], ['R²','RMSE','MAE'])):
    bars = axes[i].bar(comp['Model'], comp[key], color=colors, edgecolor='white', linewidth=1.5)
    axes[i].set_title(metric, fontweight='bold', fontsize=14)
    for bar, val in zip(bars, comp[key]):
        label = f'{val:.4f}' if key == 'R²' else format_rupiah(val)
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height(), label, ha='center', va='bottom', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=15)
plt.suptitle('Perbandingan Performa Model', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

## 4.6 Feature Importance

In [ ]:
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(feature_cols)))[::-1]
bars = ax.barh([feature_cols[i] for i in idx], importances[idx], color=colors, edgecolor='white')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance — Random Forest', fontweight='bold', fontsize=14)
ax.invert_yaxis()
for bar, val in zip(bars, importances[idx]):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4.7 Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, pred, r2) in zip(axes, [('Linear Regression',lr_pred,lr_r2), ('Random Forest',rf_pred,rf_r2), ('XGBoost',xgb_pred,xgb_r2)]):
    ax.scatter(y_test, pred, alpha=0.4, s=15, c='#3498db')
    mn, mx = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    ax.plot([mn,mx],[mn,mx],'r--', lw=2, label='Perfect')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.text(0.05,0.9,f'R²={r2:.4f}', transform=ax.transAxes, fontsize=12, fontweight='bold', bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.8))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
    ax.legend()
plt.suptitle('Actual vs Predicted', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

## 4.8 Distribusi Residual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (name, pred), c in zip(axes, [('LR',lr_pred),('RF',rf_pred),('XGB',xgb_pred)], colors):
    res = y_test.values - pred
    ax.hist(res, bins=40, color=c, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linestyle='--')
    ax.set_title(f'Residual — {name}', fontweight='bold')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
plt.tight_layout()
plt.show()

## 4.9 Simpan Model

In [ ]:
models_dir = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(models_dir, exist_ok=True)
joblib.dump(lr, os.path.join(models_dir, 'linear_regression_model.pkl'))
joblib.dump(rf, os.path.join(models_dir, 'random_forest_model.pkl'))
joblib.dump(xgb, os.path.join(models_dir, 'xgboost_model.pkl'))
joblib.dump(le, os.path.join(models_dir, 'label_encoder.pkl'))
joblib.dump(feature_cols, os.path.join(models_dir, 'feature_cols.pkl'))
print('💾 Semua model & encoder berhasil disimpan!')

---
**Selanjutnya →** `04_Regional_Analysis.ipynb`